In [140]:
# IMPORT LIBRARIES
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from geopy.distance import distance
import matplotlib.pyplot as plt
from geopy.distance import geodesic
import numpy as np
import os
import geopandas as gpd
import pandas as pd
import datetime
from sklearn.cluster import DBSCAN
from pandas.plotting import table
from matplotlib import gridspec
from matplotlib.gridspec import GridSpec
from matplotlib.colors import Normalize, LinearSegmentedColormap

In [141]:
# MANUALLY SET DATE PARAMETERS
input_end_date = '2024-09-17'
end_date = pd.to_datetime(input_end_date)

start_date = pd.to_datetime(end_date - pd.Timedelta(days=13))

print(f"Report Start Date:",start_date)
print(f"Report End Date:",end_date)

Report Start Date: 2024-09-04 00:00:00
Report End Date: 2024-09-17 00:00:00


In [142]:
# CREATE DIR FOR OUTPUTING IMAGES
save_file_path = './dengue_daily_reports_karnataka/bbmp_ward_level_hotspot/'
new_folder_name = f'bbmp-dengue-hotspots-{start_date.strftime('%b%d')}-{end_date.strftime('%b%d')}/'
save_dir = os.path.join(save_file_path, new_folder_name)
os.makedirs(save_dir, exist_ok=True)

In [143]:
# SET DATA, SHAPE FILE PATHS MANUALLY
bbmp_data_file_path = './bbmp_data/bbmp_aug24tosep17.csv'
path_assembly_shape_files = './karnataka_shape_files_taluk_district/bbmp_assembly_boundary_SHP/AC_Boundary.shp'
path_ward_shape_files = './karnataka_shape_files_taluk_district/BBMP/bbmp_final_new_wards-polygon.shp'

In [144]:
# LOAD WARD SHAPE FILES WHICH ALSO CONTAIN WARD CODE, ASSEMBLY CODE
gdf_bbmp_ward = gpd.read_file(path_ward_shape_files)

# CONVERT TO CRS
gdf_bbmp_ward_wgs84 = gdf_bbmp_ward.to_crs(epsg=4326)

# FIND CENTROIDS
gdf_bbmp_ward_wgs84['centroid'] = gdf_bbmp_ward_wgs84['geometry'].centroid
gdf_bbmp_ward_wgs84['centroid_longitude'] = gdf_bbmp_ward_wgs84['centroid'].x
gdf_bbmp_ward_wgs84['centroid_latitude'] = gdf_bbmp_ward_wgs84['centroid'].y

# EXTRACT ASSEMBLY CODE, NAME AND WARD CODE INTO SEPARATE VARIABLES
gdf_bbmp_ward_wgs84[['assembly_code', 'assembly_name']] = gdf_bbmp_ward_wgs84['assembly_1'].str.split('-', expand=True)

gdf_bbmp_ward_wgs84[['ward_code', 'ward_name']] = gdf_bbmp_ward_wgs84['proposed_w'].str.split('-', expand=True)

gdf_bbmp_ward_wgs84[['ward_code','assembly_code']] = gdf_bbmp_ward_wgs84[['ward_code','assembly_code']].astype(int)

/var/folders/0y/f_1604916677sp_j7gkzq8hc0000gn/T/ipykernel_8341/3588443719.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_bbmp_ward_wgs84['centroid'] = gdf_bbmp_ward_wgs84['geometry'].centroid


In [145]:
# LOAD ASSEMBLY SHAPE FILES

gdf_bbmp_assembly = gpd.read_file(path_assembly_shape_files)
gdf_bbmp_assembly_wgs84 = gdf_bbmp_assembly.to_crs(epsg=4326)

gdf_bbmp_assembly_wgs84= gdf_bbmp_assembly_wgs84.rename(columns={
                                                        'ASBLY_CSTN':'assembly_code',
                                                        'ASBLY_CS_1':'assembly_name'
                                                               })

In [146]:
# LOAD BBMP CASE DATA (CHECK PATH)

df_bbmp_raw = pd.read_csv(bbmp_data_file_path)

print(f"Number of cases", len(df_bbmp_raw))
print(f"Columns:", df_bbmp_raw.columns)

Number of cases 939
Columns: Index(['Case_Date', 'Patient_Name', 'mobile', 'gender', 'age', 'address',
       'Latitude', 'Longitude\n', 'test', 'date_of_reporting', 'reporting_lab',
       'ZONE', 'ward', 'ward number '],
      dtype='object')


In [147]:
# FILTER COLUMNS - MANUALLY CHECK THAT COLNAMES MATCH

df_bbmp_raw=df_bbmp_raw[['date_of_reporting', 'Latitude', 'Longitude\n', 'ZONE', 'ward', 'ward number ']]

In [148]:
# RENAME COLUMNS - MANUALLY CHECK THAT COLNAMES MATCH

df_bbmp_raw = df_bbmp_raw.rename(columns={
                'date_of_reporting' : 'result_date',
                'Latitude' : 'latitude',
                'Longitude\n' : 'longitude',
                'ZONE':'zone',
                'ward number ': 'ward_num_bbmp',
                'ward': 'ward_name_bbmp',
})

In [149]:
# CONVERT REPORT DATE TO DATETIME
df_bbmp_raw["result_date"]=pd.to_datetime(df_bbmp_raw['result_date'], format="mixed")


In [150]:
# FILTER FOR CASES IN THE REPORT PERIOD (BASED ON START AND END DATE INPUTTED)

df_bbmp_raw = df_bbmp_raw[(df_bbmp_raw['result_date'] >= start_date) \
                         & (df_bbmp_raw['result_date'] <= end_date)].reset_index(drop=True)

In [151]:
# DROP CASES WHERE REPORT DATE IS MISSING

df_bbmp_raw = df_bbmp_raw.dropna(subset=['result_date'])

In [152]:
# CLEAN LAT, LONG POSITIONS - DROP CASES WITH INVALID/MISSING LAT, LONG

df_bbmp_raw['latitude'] = df_bbmp_raw['latitude'].fillna('').astype(str).str.replace(',', '').str.strip()
df_bbmp_raw['longitude'] = df_bbmp_raw['longitude'].fillna('').astype(str).str.replace(',', '').str.strip()

df_bbmp_raw['latitude'] = pd.to_numeric(df_bbmp_raw['latitude'], errors='coerce')
df_bbmp_raw['longitude'] = pd.to_numeric(df_bbmp_raw['longitude'], errors='coerce')

df_bbmp_raw = df_bbmp_raw.dropna(subset=['latitude', 'longitude'])

df_bbmp_raw = df_bbmp_raw[(df_bbmp_raw['latitude'] >= -90) & (df_bbmp_raw['latitude'] <= 90)]
df_bbmp_raw = df_bbmp_raw[(df_bbmp_raw['longitude'] >= -180) & (df_bbmp_raw['longitude'] <= 180)]



In [153]:
# CHECK CASE COUNT

print(f"Number of cases for hotspot generation: {len(df_bbmp_raw)}")

Number of cases for hotspot generation: 568


In [154]:
# MAP BBMP LAT-LONG DATA TO SHAPE FILES

geometry = [Point(xy) for xy in zip(df_bbmp_raw['longitude'], df_bbmp_raw['latitude'])]

gdf_bbmp_last_7d = gpd.GeoDataFrame(df_bbmp_raw,crs='EPSG:4326', geometry=geometry)
gdf_bbmp_last_7d['centroid'] = gdf_bbmp_last_7d['geometry'].centroid
gdf_bbmp_last_7d['centroid_longitude_b'] = gdf_bbmp_last_7d['centroid'].x
gdf_bbmp_last_7d['centroid_latitude_b'] = gdf_bbmp_last_7d['centroid'].y


/var/folders/0y/f_1604916677sp_j7gkzq8hc0000gn/T/ipykernel_8341/2888392429.py:6: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_bbmp_last_7d['centroid'] = gdf_bbmp_last_7d['geometry'].centroid


In [155]:
# CREATE DUMMY TEST VARIABLE FOR CASE COUNTS

gdf_bbmp_last_7d['test_result'] = 1

In [156]:
# MERGE BBMP CASES WITH SHAPE FILES

gdf_cases_ward = gpd.sjoin(gdf_bbmp_last_7d, gdf_bbmp_ward_wgs84, how='left', predicate='within')

# DROP NULLS, CONVERT CODES TO INT
columns_to_check = ['result_date','test_result','latitude', 'longitude',
                    'geometry','assembly_code','assembly_name','ward_code','ward_name']


df_cases_ward = (gdf_cases_ward[~gdf_cases_ward[columns_to_check].isna().any(axis=1)]).reset_index(drop = True)
df_cases_ward[['ward_code','assembly_code']] = df_cases_ward[['ward_code','assembly_code']].astype(int)

In [157]:
# DEFINE HOTSPOT IDENTIFICATION FUNCTION

def identify_hotspots(df_cases_ward):
    gdf = gpd.GeoDataFrame(df_cases_ward, 
                           geometry=gpd.points_from_xy(df_cases_ward['longitude'], 
                                                       df_cases_ward['latitude']))
    total_cases = len(df_cases_ward)
    
    gdf.set_crs(epsg=4326, inplace=True)
    gdf = gdf.to_crs(epsg=32643)  # Example for UTM zone 43N, adjust as necessary
    coords = np.array(list(gdf.geometry.apply(lambda geom: (geom.x, geom.y))))
    
    db = DBSCAN(eps=100, min_samples=1, metric='euclidean').fit(coords)
    gdf['cluster'] = db.labels_
    
    filtered_gdf = gdf[gdf['cluster'] != -1]
    valid_clusters = filtered_gdf.groupby('cluster').filter(lambda x: len(x) > 1)['cluster'].nunique()
    
    valid_cluster_labels = filtered_gdf.groupby('cluster').filter(lambda x: len(x) > 1)['cluster'].unique()
    highlighted_gdf = gdf[gdf['cluster'].isin(valid_cluster_labels)]
    
    return gdf, total_cases, valid_clusters, highlighted_gdf

In [160]:
# ITERATE THROUGH ASSEMBLY CODES AND GENERATE HOTSPOT MAPS
assembly_codes = gdf_bbmp_assembly_wgs84['assembly_code'].unique()


In [161]:

df_assembly_summary = pd.DataFrame()

for assembly_code in assembly_codes:
    df_cases_ward_filt = df_cases_ward[df_cases_ward['assembly_code']== assembly_code]
    
    gdf_bbmp_ward_wgs84_filt = gdf_bbmp_ward_wgs84[gdf_bbmp_ward_wgs84['assembly_code']==assembly_code]
    gdf_bbmp_assembly_wgs84_filt  = gdf_bbmp_assembly_wgs84[gdf_bbmp_assembly_wgs84['assembly_code']==assembly_code]
    df_cases_ward_filt_cases = df_cases_ward_filt.groupby(['ward_name', 'ward_code'])\
                                                      ['test_result'].sum().reset_index()

    ticks = []
    tick_labels = []
    ward_codes = df_cases_ward_filt_cases['ward_code'].unique()
    results = []
    
    # fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    fig = plt.figure(figsize=(18, 8))
    gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1.25])

    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])
    
    gdf_bbmp_assembly_wgs84_filt = gdf_bbmp_assembly_wgs84_filt.to_crs(epsg=32643)
    gdf_bbmp_assembly_wgs84_filt.plot(ax=ax1, color='white', edgecolor=(0.2, 0.2, 0.8), 
                        alpha=0.8, linewidth=1)  
    
    # Iterate through wards within the AC
    for ward_code in ward_codes:
        df_cases_ward_sel = df_cases_ward_filt[df_cases_ward_filt['ward_code'] == ward_code]
        gdf_bbmp_ward_wgs84_sel = gdf_bbmp_ward_wgs84[gdf_bbmp_ward_wgs84['ward_code'] == ward_code]
        ward_name = gdf_bbmp_ward_wgs84_sel['ward_name'].values[0]
        assembly_name = gdf_bbmp_ward_wgs84_sel['assembly_name'].values[0]
        assembly_code = gdf_bbmp_ward_wgs84_sel['assembly_code'].values[0]

        # call function to identify hotspots
        gdf, total_cases_ward_sel, valid_clusters, highlighted_gdf = identify_hotspots(df_cases_ward_sel)

        results.append({
            'Assembly Code': assembly_code,
            'Assembly Name': assembly_name,
            'Ward Code': ward_code,
            'Ward Name': ward_name,
            'Total Cases': total_cases_ward_sel,
            'Number of Hotspots': valid_clusters
        })
        
        df_ward_summary = pd.DataFrame(results)

        gdf_bbmp_ward_wgs84_filt = gdf_bbmp_ward_wgs84_filt.to_crs(epsg=f'32643')
        df_ward_case_hotspot = pd.merge(df_ward_summary, gdf_bbmp_ward_wgs84_filt, 
        left_on=['Assembly Code', 'Assembly Name', 'Ward Code','Ward Name'], 
        right_on=['assembly_code', 'assembly_name', 'ward_code','ward_name'], 
        how='inner')
        
        gdf_bbmp_ward_wgs84_sel = gdf_bbmp_ward_wgs84_sel.to_crs(epsg=f'32643')
        gdf_bbmp_ward_wgs84_sel.plot(ax=ax1, color='white', edgecolor=(0.2, 0.4, 0.8), 
                                     alpha=0.8, linewidth=0.6)

        hotspot_color = (255/255, 0/255, 0/255)
        # ax.scatter(gdf.geometry.x, gdf.geometry.y, color=(255/255, 176/255, 66/255), alpha=0.8, marker='o', s=15, edgecolor='black', linewidth=0.3, label='Cases')
        ax1.scatter(highlighted_gdf.geometry.x, highlighted_gdf.geometry.y, 
                   color=hotspot_color, marker='o', alpha=1, s=30, edgecolor='black', 
                   linewidth=0.2, label='Hotspot(cases<100m)')

        for idx, row in gdf_bbmp_ward_wgs84_sel.iterrows():
            # Define the annotation parameters
            zone_name = row['ward_code']  # Replace 'ZoneName' with your actual column name
            x, y = row.geometry.centroid.x, row.geometry.centroid.y  # Centroid of the polygon
    
            # Add the annotation
            ax1.annotate(zone_name, xy=(x, y), xytext=(3, 3), 
                textcoords='offset points', fontsize=6, 
                color=(0.0, 0, 0.0, 1),ha='center')

        total_cases = df_ward_summary['Total Cases'].sum()
        total_hotspot = df_ward_summary['Number of Hotspots'].sum()

        # ax1.set_title(textstr, fontsize=12, loc='left')
        ax1.set_xticklabels([])
        ax1.set_yticklabels([])
        ax1.tick_params(axis='both', which='both', length=0)
    
#####################  Get assembly level summary ##########################

    df_assembly_counts = df_ward_summary.groupby(['Assembly Name','Assembly Code'])\
                                            [['Total Cases','Number of Hotspots']].sum().reset_index()        
    df_assembly_summary = pd.concat([df_assembly_summary, df_assembly_counts], ignore_index=True)
    

    
################################## COLOR MAP DENGUE CASES##################################

    # gdf_bbmp_ward_wgs84_ward_filt = gdf_bbmp_ward_wgs84_ward_filt.to_crs(epsg=f'32643')
    gdf_bbmp_ward_wgs84_filt = gdf_bbmp_ward_wgs84_filt.to_crs(epsg=f'32643')
    # cases_min = df_ward_case_hotspot['Total Cases'].min()
    cases_min = 0
    cases_max = df_ward_case_hotspot['Total Cases'].max()
    norm = Normalize(vmin=cases_min, vmax=cases_max)

    colors = [
        (255/255, 245/255, 235/255, 0.6),  
        (254/255, 230/255, 206/255, 0.6),   
        (253/255, 208/255, 162/255, 0.6),   
        (253/255, 174/255, 107/255, 0.6), 
        (253/255, 141/255, 60/255, 0.6)
    ]
  
    if cases_max < 3 & cases_max > 1:
        ticks = [0, 1, 2]
        cmap = LinearSegmentedColormap.from_list('custom_cmap', colors, N=2)
        tick_labels = [f'{int(t)}' for t in ticks]
        
    elif cases_max < 4 & cases_max > 3:
        ticks = [0, 1, 2, 3]
        cmap = LinearSegmentedColormap.from_list('custom_cmap', colors, N=3)
        tick_labels = [f'{int(t)}' for t in ticks]
        
    elif cases_max < 5 & cases_max > 4 :
        ticks = [0, 1, 2, 3, 4]
        cmap = LinearSegmentedColormap.from_list('custom_cmap', colors, N=4)
        tick_labels = [f'{int(t)}' for t in ticks]

    else:
        ticks = [cases_min + (i * (cases_max - cases_min) / 5) for i in range(6)]
        cmap = LinearSegmentedColormap.from_list('custom_cmap', colors, N=5)
        tick_labels = [f'{int(t)}' for t in ticks]
      
    for idx, row in df_ward_case_hotspot.iterrows():
        cases = row['Total Cases']
        color = cmap(norm(cases))
        gdf_bbmp_ward_wgs84_filt.loc[
        gdf_bbmp_ward_wgs84_filt['ward_code'] == row['ward_code']].plot(
        ax=ax1, color=color, edgecolor=(0.5, 0.5, 0.5), 
        linewidth=0.4, alpha=0.5)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax1, orientation='vertical', shrink=1)
  
    cbar.set_ticks(ticks)
    cbar.set_ticklabels(tick_labels)
    cbar.ax.tick_params(labelsize=16)
    cbar.ax.set_title('cases', fontsize=14, pad=20)
    pos = ax1.get_position()
    left, bottom, width, height = pos.x0, pos.y0, pos.width, pos.height
    cbar.ax.set_position([left + width + 0.03, bottom, 0.02, height])  # [left, bottom, width, height]

################################### COLOR MAP DENGUE CASES ##################################

##################################### SUMMARY TABLE ####################################

    columns_to_keep = ['Ward Code', 'Ward Name',
       'Total Cases', 'Number of Hotspots']

    df_summary_filt = df_ward_summary[columns_to_keep]
    total_cases = df_summary_filt['Total Cases'].sum()
    total_hotspot = df_summary_filt['Number of Hotspots'].sum()

    df_summary_filt = df_summary_filt.rename(columns={
    'Ward Code': 'Ward \nCode',
    'Total Cases':'Total \nCases',
    'Number of Hotspots': 'Number of \nHotspots',
    })

    # Hide the axes
    ax2.xaxis.set_visible(False)
    ax2.yaxis.set_visible(False)
    ax2.set_frame_on(False)

    # Create a table
    table = ax2.table(cellText=df_summary_filt.values,
                     colLabels=df_summary_filt.columns,
                     cellLoc='center', loc='center')

    col_widths = [0.2, 0.38, 0.2, 0.2]  # Adjust these values based on your data
   
    for i, width in enumerate(col_widths):
        for j in range(len(df_summary_filt) + 1):  # +1 for the header
            cell = table[j, i]
            cell.set_width(width)
            if j == 0:
                cell.set_height(0.2)  # Adjust the header height as needed
            else:
                cell.set_height(0.075)  # Adjust the row height as needed

    table.auto_set_font_size(False)
    table.set_fontsize(14)
    table.scale(1, 1)  # Adjust scaling as needed

    table_edge_color = (150/255, 150/255, 150/255,0.7) 
    cell_dict = table.get_celld()
    for (i, j), cell in cell_dict.items():
        cell.set_edgecolor(table_edge_color)
        cell.set_linewidth(0.3)  # Adjust this value to change line width

# Set Sans-Serif font family for all cells
    for key, cell in table.get_celld().items():
        cell.set_text_props(fontfamily='sans-serif')


    # ax2.set_title(title_str, fontsize=14,pad=0,loc='center')

    title_str = '\n'.join((f"Date: {start_date.strftime('%b %d')} to {end_date.strftime('%b %d')}",
            f"Assembly Name: {assembly_name}\n",
            f"Total Cases: {total_cases}\n",
            f"Total Number of Hotspots: {total_hotspot} (Cases within 100m radius)\n",
            ))
    
    fig.suptitle(title_str,fontsize=14,fontweight='bold')


    plt.tight_layout(rect=[0, 0, 0.95, 0.95])  # Adjust layout to make room for the main title
   
    
    save_hotspot_table = f'{save_dir}bbmp_{assembly_name}_cases_hotspot_{end_date.strftime('%b %d')}.png'
    
    plt.savefig(save_hotspot_table, dpi=300, bbox_inches='tight',pad_inches=0.5)

    plt.close(fig)

################### TABLE ASSEMBLY LEVEL SUMMARY #################################

df_assembly_summary_filt  = df_assembly_summary.copy()
columns_to_keep = ['Assembly Code','Assembly Name',
       'Total Cases', 'Number of Hotspots']

total_cases_assembly = df_assembly_summary_filt['Total Cases'].sum()
total_hotspot_assembly = df_assembly_summary_filt['Number of Hotspots'].sum()

df_assembly_summary_filt = df_assembly_summary_filt[columns_to_keep]\
                                .rename(columns={'Assembly Code':'Assembly \nCode',
                                                 'Assembly Name':'Assembly \nName',
                                                 'Total Cases':'Total \nCases',
                                                 'Number of Hotspots': 'Number of \nHotspots'
                                                })

fig, ax3 = plt.subplots(figsize=(8, 10))

ax3.xaxis.set_visible(False)
ax3.yaxis.set_visible(False)
ax3.set_frame_on(False)

# Create a table
table = ax3.table(cellText=df_assembly_summary_filt.values,
                     colLabels=df_assembly_summary_filt.columns,
                     cellLoc='center', loc='center')

col_widths = [0.2, 0.45, 0.2, 0.2]  # Adjust these values based on your data
   
for i, width in enumerate(col_widths):
    for j in range(len(df_assembly_summary_filt) + 1):  # +1 for the header
        cell = table[j, i]
        cell.set_width(width)
        if j == 0:
            cell.set_height(0.1)  # Adjust the header height as needed
        else:
            cell.set_height(0.06)  # Adjust the row height as needed

table.auto_set_font_size(False)
table.set_fontsize(14)
table.scale(1, 1.25)  # Adjust scaling as needed

table_edge_color = (150/255, 150/255, 150/255,0.7) 
cell_dict = table.get_celld()
for (i, j), cell in cell_dict.items():
    cell.set_edgecolor(table_edge_color)
    cell.set_linewidth(0.3)  # Adjust this value to change line width

# Set Sans-Serif font family for all cells
for key, cell in table.get_celld().items():
    cell.set_text_props(fontfamily='sans-serif')


title_str = '\n'.join((f"Date: {start_date.strftime('%b %d')} to {end_date.strftime('%b %d')}",
            f"BBMP Total Cases: {total_cases_assembly}",
            f"BBMP Total Number of Hotspots: {total_hotspot_assembly}",
            f""
            ))
    
plt.title(title_str,fontsize=14,fontweight='bold',y=1.8)

plt.tight_layout(rect=[0, 0, 0.95, 0.95])  # Adjust layout to make room for the main title
   
save_hotspot_table = f'{save_dir}bbmp_assembly_summary_cases_hotspot_{end_date.strftime('%b %d')}.png'
    
plt.savefig(save_hotspot_table, dpi=300, bbox_inches='tight',pad_inches=0.5)

plt.close(fig)


/var/folders/0y/f_1604916677sp_j7gkzq8hc0000gn/T/ipykernel_8341/120868004.py:132: UserWarning: The GeoDataFrame you are attempting to plot is empty. Nothing has been displayed.
  gdf_bbmp_ward_wgs84_filt['ward_code'] == row['ward_code']].plot(
/var/folders/0y/f_1604916677sp_j7gkzq8hc0000gn/T/ipykernel_8341/120868004.py:132: UserWarning: The GeoDataFrame you are attempting to plot is empty. Nothing has been displayed.
  gdf_bbmp_ward_wgs84_filt['ward_code'] == row['ward_code']].plot(
/var/folders/0y/f_1604916677sp_j7gkzq8hc0000gn/T/ipykernel_8341/120868004.py:132: UserWarning: The GeoDataFrame you are attempting to plot is empty. Nothing has been displayed.
  gdf_bbmp_ward_wgs84_filt['ward_code'] == row['ward_code']].plot(
/var/folders/0y/f_1604916677sp_j7gkzq8hc0000gn/T/ipykernel_8341/120868004.py:132: UserWarning: The GeoDataFrame you are attempting to plot is empty. Nothing has been displayed.
  gdf_bbmp_ward_wgs84_filt['ward_code'] == row['ward_code']].plot(
/var/folders/0y/f_160491

In [163]:
df_cases_ward[['result_date', 'latitude', 'longitude', 'zone', 
       'assembly_code', 'assembly_name', 'ward_code', 'ward_name']].to_csv(f"{save_dir}cases_by_ward.csv", index=False)